# CCKW 滤波分析 - 比湿数据 (Specific Humidity)

本脚本对比湿数据(hus)进行Kelvin波滤波分析

**处理流程:**
1. 加载3D比湿数据 (time, lev, lat, lon)
2. 对每个垂直层分别应用Kelvin波滤波
3. 保存滤波后的数据到缓存，方便后续直接读取

**日期:** 2026.02.04

In [18]:
# Load packages
import xarray as xr
import numpy as np
import os
import time
import sys
import gc
from pathlib import Path

WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools/")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))

from wave_tools.filters import CCKWFilter

print("="*70)
print("✅ Packages loaded successfully")
print("="*70)

# Set up directories
CACHE_DIR = "./cache/kelvin_wave_3d/"
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"📁 Cache directory: {CACHE_DIR}")
print("="*70)

✅ Packages loaded successfully
📁 Cache directory: ./cache/kelvin_wave_3d/


In [19]:
# 辅助函数：检查处理进度
def check_processing_status():
    """检查各实验的处理进度"""
    print("="*70)
    print("📊 Checking Processing Status")
    print("="*70)
    
    for exp in ['CNTL', 'P4K', '4CO2']:
        print(f"\n📍 {exp}:")
        
        # 检查最终文件
        final_file = os.path.join(CACHE_DIR, f'kelvin_hus_{exp.lower()}.nc')
        if os.path.exists(final_file):
            print(f"  ✅ Final merged file exists")
            data = xr.open_dataarray(final_file)
            print(f"     Shape: {data.shape}")
            print(f"     Size: {os.path.getsize(final_file) / 1e9:.2f} GB")
            continue
        
        # 检查单层文件
        level_dir = os.path.join(CACHE_DIR, f'hus_levels_{exp.lower()}')
        if os.path.exists(level_dir):
            level_files = [f for f in os.listdir(level_dir) if f.endswith('.nc')]
            print(f"  🔄 Partial processing: {len(level_files)} level files found")
            print(f"     Can resume from last checkpoint")
        else:
            print(f"  ⚪ Not started")
    
    print("\n" + "="*70)

print("✅ Helper function loaded: check_processing_status()")
print("   Usage: Call check_processing_status() to see progress")

✅ Helper function loaded: check_processing_status()
   Usage: Call check_processing_status() to see progress


In [20]:
# Step 1: 加载比湿数据 (hus)
print("="*70)
print("📊 Loading specific humidity (hus) data")
print("="*70)

# 定义实验和目录映射
exp_dir_map = {
    'CNTL': 'cntl',
    'P4K': 'p4k',
    '4CO2': '4co2'
}

# 数据存储字典
hus_data = {}

for exp, dir_name in exp_dir_map.items():
    print(f"\n{'='*60}")
    print(f"📍 Loading {exp} data")
    print(f"{'='*60}")
    
    # 加载 hus 数据
    hus_file = f"/work/mh1498/m301257/3D_data/{dir_name}/hus_all_levels.nc"
    try:
        hus_ds = xr.open_dataset(hus_file)
        hus_data[exp] = hus_ds['hus'] if 'hus' in hus_ds else hus_ds[list(hus_ds.data_vars)[0]]
        print(f"  ✅ Hus loaded successfully")
        print(f"     Shape: {hus_data[exp].shape}")
        print(f"     Dims: {hus_data[exp].dims}")
        print(f"     Levels: {len(hus_data[exp].lev) if 'lev' in hus_data[exp].dims else 'N/A'}")
    except Exception as e:
        print(f"  ❌ Hus: Failed - {str(e)}")   

print("\n✅ All data loaded successfully")
print("="*70)

📊 Loading specific humidity (hus) data

📍 Loading CNTL data
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

📍 Loading P4K data
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

📍 Loading 4CO2 data
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

✅ All data loaded successfully


In [21]:
# 检查数据结构（调试用）
print("="*70)
print("🔍 Inspecting data structure")
print("="*70)

for exp in ['CNTL']:  # 只检查第一个实验
    if exp in hus_data:
        data = hus_data[exp]
        print(f"\n📍 {exp}:")
        print(f"  Type: {type(data)}")
        print(f"  Name: {data.name}")
        print(f"  Dims: {data.dims}")
        print(f"  Coords: {list(data.coords.keys())}")
        print(f"  Shape: {data.shape}")
        
        # 测试选择单层
        if 'level' in data.dims:
            test_level = data.level.values[0]
            test_data = data.sel(level=test_level)
            print(f"\n  Test single level ({test_level}):")
            print(f"    Type: {type(test_data)}")
            print(f"    Name: {test_data.name}")
            print(f"    Dims: {test_data.dims}")
            print(f"    Shape: {test_data.shape}")

print("\n" + "="*70)

🔍 Inspecting data structure

📍 CNTL:
  Type: <class 'xarray.core.dataarray.DataArray'>
  Name: hus
  Dims: ('level', 'time', 'lat', 'lon')
  Coords: ['level', 'time', 'lat', 'lon']
  Shape: (22, 5114, 15, 180)

  Test single level (31):
    Type: <class 'xarray.core.dataarray.DataArray'>
    Name: hus
    Dims: ('time', 'lat', 'lon')
    Shape: (5114, 15, 180)



In [22]:
# Step 2: 对比湿数据应用Kelvin波滤波（逐层处理，每层单独保存）
print("="*70)
print("🌊 Applying Kelvin wave filter to specific humidity data")
print("   Processing each vertical level separately")
print("   💾 Each level saved individually to prevent data loss")
print("="*70)

# 滤波参数设置
sel_dict = {
    'time': slice('1980-01-01', '1993-12-31'),
    'lat': slice(-15, 15)
}
wave_name = 'kelvin'
units = 'kg/kg'

# 存储滤波后的数据
kelvin_hus = {}

total_start_time = time.time()

for exp_idx, exp in enumerate(['CNTL', 'P4K', '4CO2'], 1):
    print(f"\n{'='*60}")
    print(f"📍 Processing {exp} ({exp_idx}/3)")
    print(f"{'='*60}")
    
    # 最终合并文件
    final_cache_file = os.path.join(CACHE_DIR, f'kelvin_hus_{exp.lower()}.nc')
    
    # 检查最终文件是否已存在
    if os.path.exists(final_cache_file):
        print(f"  ♻️  Final merged file already exists, loading from cache...")
        ds = xr.open_dataset(final_cache_file, chunks={'time': 1000})
        kelvin_hus[exp] = ds['hus'] if 'hus' in ds else ds[list(ds.data_vars)[0]]
        print(f"  ✅ Loaded! Shape: {kelvin_hus[exp].shape}")
        continue
    
    # 如果最终文件不存在，检查并处理各层
    exp_start_time = time.time()
    
    try:
        # 获取垂直层数
        hus_exp = hus_data[exp]
        levels = hus_exp.level.values
        n_levels = len(levels)
        
        print(f"  📊 Total levels to process: {n_levels}")
        
        # 为每层创建单独的缓存目录
        level_cache_dir = os.path.join(CACHE_DIR, f'hus_levels_{exp.lower()}')
        os.makedirs(level_cache_dir, exist_ok=True)
        
        # 检查已处理的层
        processed_levels = []
        for lev in levels:
            level_file = os.path.join(level_cache_dir, f'level_{int(lev):03d}.nc')
            if os.path.exists(level_file):
                processed_levels.append(lev)
        
        if processed_levels:
            print(f"  ♻️  Found {len(processed_levels)}/{n_levels} already processed levels")
        
        print(f"  ⏳ Starting level-by-level filtering...")
        
        # 逐层进行滤波
        for lev_idx, lev in enumerate(levels, 1):
            level_file = os.path.join(level_cache_dir, f'level_{int(lev):03d}.nc')
            
            # 检查该层是否已处理
            if os.path.exists(level_file):
                print(f"\n    ⏭️  Level {lev_idx}/{n_levels} (level={int(lev)}): Already processed, skipping")
                continue
            
            print(f"\n    🔹 Level {lev_idx}/{n_levels} (level={int(lev)})")
            
            try:
                # 提取当前层数据
                hus_level = hus_exp.sel(level=lev)
                
                # 初始化滤波器
                wave_filter = CCKWFilter(
                    ds=hus_level,
                    sel_dict=sel_dict,
                    wave_name=wave_name,
                    units=units,
                    spd=1,
                    n_workers=2,  # 减少worker数量以节省内存
                    verbose=False
                )
                
                # 执行滤波步骤
                wave_filter.load_data()
                wave_filter.detrend_data()
                wave_filter.fft_transform()
                wave_filter.apply_filter()
                wave_filter.inverse_fft()
                filtered_level = wave_filter.create_output()
                
                # 添加level维度
                filtered_level = filtered_level.expand_dims({'level': [lev]})
                
                # 转换为Dataset并保存（确保有变量名）
                if not filtered_level.name:
                    filtered_level.name = 'hus'
                
                # 转换为Dataset
                ds_to_save = filtered_level.to_dataset()
                encoding = {'hus': {'zlib': True, 'complevel': 4}}
                ds_to_save.to_netcdf(level_file, encoding=encoding)
                print(f"       💾 Saved to: {os.path.basename(level_file)}")
                
                print(f"       ✅ Completed")
                
                # 清理内存
                del wave_filter, filtered_level, hus_level
                gc.collect()
                
            except Exception as e:
                print(f"       ❌ Error at level {lev}: {str(e)}")
                import traceback
                traceback.print_exc()
                continue
        
        # 所有层处理完后，合并成一个文件
        print(f"\n  🔄 Merging all processed levels...")
        
        # 收集所有已处理的层文件
        level_files = sorted([
            os.path.join(level_cache_dir, f) 
            for f in os.listdir(level_cache_dir) 
            if f.endswith('.nc')
        ])
        
        if len(level_files) > 0:
            print(f"     Found {len(level_files)} level files to merge")
            
            # 逐个加载并合并（节省内存）
            filtered_levels = []
            for lf in level_files:
                try:
                    # 从Dataset加载，提取hus变量
                    ds = xr.open_dataset(lf)
                    data = ds['hus'] if 'hus' in ds else ds[list(ds.data_vars)[0]]
                    filtered_levels.append(data)
                except Exception as e:
                    print(f"     ⚠️  Failed to load {os.path.basename(lf)}: {str(e)}")
            
            if filtered_levels:
                # 合并所有层
                filtered_data = xr.concat(filtered_levels, dim='level')
                
                # 确保有变量名并转换为Dataset保存
                print(f"  💾 Saving final merged file...")
                if not filtered_data.name:
                    filtered_data.name = 'hus'
                
                ds_to_save = filtered_data.to_dataset()
                encoding = {'hus': {'zlib': True, 'complevel': 4}}
                ds_to_save.to_netcdf(final_cache_file, encoding=encoding)
                
                kelvin_hus[exp] = filtered_data
                
                exp_elapsed = time.time() - exp_start_time
                print(f"  ✅ {exp} completed in {exp_elapsed/60:.1f} minutes")
                print(f"     Shape: {filtered_data.shape}")
                print(f"     Memory: {filtered_data.nbytes / 1e9:.2f} GB")
                print(f"     Saved to: {os.path.basename(final_cache_file)}")
                
                # 清理内存
                del filtered_data, filtered_levels, ds_to_save
                gc.collect()
            else:
                print(f"  ❌ No valid level files could be loaded")
        else:
            print(f"  ❌ No level files found for {exp}")
            
    except Exception as e:
        print(f"  ❌ Error processing {exp}: {str(e)}")
        import traceback
        traceback.print_exc()
    
    # 每个实验后清理内存
    if exp in hus_data:
        del hus_data[exp]
    gc.collect()

total_elapsed = time.time() - total_start_time
print("\n" + "="*70)
print(f"✅ All filtering completed in {total_elapsed/60:.1f} minutes")
print(f"   Processed {len(kelvin_hus)} experiments successfully")
print("="*70)

🌊 Applying Kelvin wave filter to specific humidity data
   Processing each vertical level separately
   💾 Each level saved individually to prevent data loss

📍 Processing CNTL (1/3)
  📊 Total levels to process: 22
  ⏳ Starting level-by-level filtering...

    🔹 Level 1/22 (level=31)
  📊 Total levels to process: 22
  ⏳ Starting level-by-level filtering...

    🔹 Level 1/22 (level=31)
       💾 Saved to: level_031.nc
       ✅ Completed

    🔹 Level 2/22 (level=35)
       💾 Saved to: level_031.nc
       ✅ Completed

    🔹 Level 2/22 (level=35)
       💾 Saved to: level_035.nc
       ✅ Completed

    🔹 Level 3/22 (level=38)
       💾 Saved to: level_035.nc
       ✅ Completed

    🔹 Level 3/22 (level=38)
       💾 Saved to: level_038.nc
       ✅ Completed

    🔹 Level 4/22 (level=41)
       💾 Saved to: level_038.nc
       ✅ Completed

    🔹 Level 4/22 (level=41)
       💾 Saved to: level_041.nc
       ✅ Completed

    🔹 Level 5/22 (level=46)
       💾 Saved to: level_041.nc
       ✅ Completed

  

/tmp/ipykernel_3177283/3226634315.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(final_cache_file, chunks={'time': 1000})
/tmp/ipykernel_3177283/3226634315.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(final_cache_file, chunks={'time': 1000})


In [23]:
# Step 3: 验证缓存的滤波数据
print("="*70)
print("🔍 Verifying cached Kelvin-filtered hus data")
print("="*70)

for exp in ['CNTL', 'P4K', '4CO2']:
    cache_file = os.path.join(CACHE_DIR, f'kelvin_hus_{exp.lower()}.nc')
    
    if os.path.exists(cache_file):
        print(f"\n📍 {exp}:")
        try:
            # 加载数据（从Dataset中提取）
            ds = xr.open_dataset(cache_file)
            data = ds['hus'] if 'hus' in ds else ds[list(ds.data_vars)[0]]
            
            # 打印基本信息
            print(f"  ✅ File exists and readable")
            print(f"     Variable: {data.name}")
            print(f"     Shape: {data.shape}")
            print(f"     Dims: {data.dims}")
            print(f"     Levels: {len(data.level) if 'level' in data.dims else 'N/A'}")
            print(f"     Level range: {float(data.level.min()):.1f} - {float(data.level.max()):.1f}")
            print(f"     File size: {os.path.getsize(cache_file) / 1e9:.2f} GB")
            
            # 检查数据范围
            print(f"     Data range: [{float(data.min()):.6e}, {float(data.max()):.6e}]")
            
        except Exception as e:
            print(f"  ❌ Error reading file: {str(e)}")
    else:
        print(f"\n❌ {exp}: Cache file not found")
        print(f"   Expected: {cache_file}")



🔍 Verifying cached Kelvin-filtered hus data

📍 CNTL:
  ✅ File exists and readable
     Variable: hus
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: 22
     Level range: 31.0 - 90.0
     File size: 2.07 GB
  ✅ File exists and readable
     Variable: hus
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: 22
     Level range: 31.0 - 90.0
     File size: 2.07 GB
     Data range: [-2.780857e-03, 2.342525e-03]

📍 P4K:
     Data range: [-2.780857e-03, 2.342525e-03]

📍 P4K:
  ✅ File exists and readable
     Variable: hus
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: 22
     Level range: 31.0 - 90.0
     File size: 2.07 GB
  ✅ File exists and readable
     Variable: hus
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: 22
     Level range: 31.0 - 90.0
     File size: 2.07 GB
     Data range: [-2.576761e-03, 2.548486e-03]

📍 4CO2:
     Data